In [15]:
import os
import geopandas as gpd
import numpy as np
import xarray as xr
import torch

from rasterio.features import rasterize
from rasterio.transform import from_origin
import rasterio.enums
from collections import defaultdict
from shapely.validation import make_valid

In [23]:
# ============================================================
# PARÁMETROS (todos juntos, al principio)
# ============================================================
shapefile_path = "/home/angelo/MyStore/Proyecto Investigador/2017/shp_2017/shp_2017.shp"
netcdf_path = "/temusa_store/Processed_Data/Sentinel1/Sevilla/SLC/Dataset_2017-21.nc"

AREA_MIN_HA = 0.2
AREA_MIN_M2 = AREA_MIN_HA * 10_000        # 0.2 ha = 2000 m²
CRS_METRICO = "EPSG:25830"                # ETRS89 / UTM 30N (metros), solo para medir el área
COL_CULTIVO = "DS_CULTIVO"                # columna del shapefile con el cultivo
CULTIVOS_EXCLUIDOS = {"Rice"}             # valores exactos de esa columna
ALL_TOUCHED = True                        # incluye píxeles que solo tocan el polígono

# ============================================================
# 1. CARGAR MAPA VECTORIAL (SHAPEFILE)
# ============================================================
gdf = gpd.read_file(shapefile_path)


gdf = gpd.read_file(shapefile_path)

gdf = gpd.read_file(shapefile_path)

# El shapefile contiene 635 geometrías nulas (sin polígono).
# El filtro de área las elimina automáticamente ya que su área es NaN.
# No hay polígonos corruptos, por lo que no se aplica ninguna reparación.

# ============================================================
# 2. ID NUMÉRICO POR PARCELA (ANTES de filtrar: la numeración no cambia)
# ============================================================
gdf["burnid_num"] = range(1, len(gdf) + 1)
# ============================================================
# 3. ÁREA EN m² Y FILTROS
# ============================================================
assert gdf.crs is not None, "El shapefile no tiene CRS: no se puede medir el área"
# El área se mide en un CRS métrico; la geometría original no se toca
# (el ráster usa el CRS del shapefile).
gdf["area_m2"] = gdf.to_crs(CRS_METRICO).geometry.area

en_area = gdf["area_m2"] > AREA_MIN_M2
es_excl = gdf[COL_CULTIVO].isin(CULTIVOS_EXCLUIDOS)

gdf_filtrado = gdf[en_area & ~es_excl].copy()
n_antes, n_despues = len(gdf), len(gdf_filtrado)

print("=== FILTROS ===")
print(f"Umbral de área: > {AREA_MIN_HA} ha ({AREA_MIN_M2:.0f} m²)")
print(f"Cultivos excluidos: {sorted(CULTIVOS_EXCLUIDOS)}")
print(f"Antes: {n_antes}")
print(f"Eliminadas por área (<= {AREA_MIN_HA} ha): {(~en_area).sum()}")
print(f"Eliminadas por cultivo (con área suficiente): {(en_area & es_excl).sum()}")
print(f"Después: {n_despues}")

# ============================================================
# 4. METADATOS DEL LIENZO (IMAGEN DE SATÉLITE)
# ============================================================
ds = xr.open_dataset(netcdf_path)

x = ds["x"].values
y = ds["y"].values
res_x = x[1] - x[0]
res_y = y[0] - y[1]

# Origen corregido: un píxel hacia el sureste respecto a (x.min(), y.max()).
# Verificado comparando con el crop_type_map del NetCDF: concordancia 0.9999
# en el interior de las parcelas con este origen (y 0.9646 con el anterior).
transform = from_origin(x.min() + res_x, y.max() - res_y, res_x, res_y)
alto_matriz = len(y)
ancho_matriz = len(x)
ds.close()
# ============================================================
# 5. RASTERIZACIÓN (solo parcelas que pasan los filtros)
# ============================================================
# Mayor a menor área: las grandes van abajo y las pequeñas encima
gdf_ordenado = gdf_filtrado.sort_values(by="area_m2", ascending=False)

lista_shapes = [(geom, int(num))
                for geom, num in zip(gdf_ordenado.geometry, gdf_ordenado["burnid_num"])]

# Los píxeles de parcelas eliminadas quedan en 0 (fondo); no se reasignan a vecinas
mascara_parcelas = rasterize(
    shapes=lista_shapes,
    out_shape=(alto_matriz, ancho_matriz),
    transform=transform,
    fill=0,
    dtype=np.int32,
    all_touched=ALL_TOUCHED,
    merge_alg=rasterio.enums.MergeAlg.replace,
)

# ============================================================
# 6. DIAGNÓSTICOS
# ============================================================
print("\n=== DIAGNÓSTICO DE LA RASTERIZACIÓN ===")
print(f"Polígonos en el shapefile: {n_antes} | tras filtros: {n_despues}")
print(f"Dimensiones de la matriz ráster: {mascara_parcelas.shape}")
ids_pintados = np.unique(mascara_parcelas)
ids_pintados = ids_pintados[ids_pintados != 0]
print(f"Parcelas pintadas: {len(ids_pintados)} (esperadas: {n_despues})")
if len(ids_pintados) != n_despues:
    print(f"  AVISO: {n_despues - len(ids_pintados)} parcelas sin ningún píxel "
          f"(tapadas por otras o fuera de la imagen)")

=== FILTROS ===
Umbral de área: > 0.2 ha (2000 m²)
Cultivos excluidos: ['Rice']
Antes: 4610
Eliminadas por área (<= 0.2 ha): 710
Eliminadas por cultivo (con área suficiente): 291
Después: 3609

=== DIAGNÓSTICO DE LA RASTERIZACIÓN ===
Polígonos en el shapefile: 4610 | tras filtros: 3609
Dimensiones de la matriz ráster: (3587, 3607)
Parcelas pintadas: 3609 (esperadas: 3609)


In [27]:
# ============================================================
# CONFIGURACIÓN DE RUTAS, ESTRUCTURAS Y DICCIONARIO BASE
# ============================================================
# Semilla para reproducibilidad
RNG = np.random.default_rng(42)

ruta_carpeta_npz = "/home/angelo/MyStore/Proyecto Investigador/2017/series_temporales_2017_normalized(backscatter)/"


clases_cultivo = {
    "None": 0,
    "Alfalfa": 1,
    "Barley": 2,
    "Carrot": 3,
    "Chickpea": 4,
    "Cotton": 5,
    "Fallow": 6,
    "Maize": 7,
    "Oats": 8,
    "Onion": 9,
    "Pepper": 10,
    "Potato": 11,
    "Pumpkin": 12,
    "Quinoa": 13,
    "Rice": 14,
    "SugarBeet": 15,
    "Sunflower": 16,
    "SweetPotato": 17,
    "Tomato": 18,
    "Wheat": 19
}

PROP_TRAIN = 0.50
PROP_TEST = 0.50

datos_train_por_cultivo = {}
datos_test_por_cultivo = {}

reporte_division = {}

# ============================================================
# PASADA 1 — CARGA DE TODOS LOS CULTIVOS Y CONTEO GLOBAL POR PARCELA
# ============================================================
# Aquí NO se decide train/test todavía. Antes de repartir nada, se
# recorren todos los cultivos y se acumula, por parcela, cuántos píxeles
# tiene en CADA cultivo en el que aparece.
#
# Esto es necesario porque intervienen DOS capas espaciales distintas y
# no perfectamente coincidentes:
#   · crop_type_map  -> de donde se extraen las series (etiqueta de píxel)
#   · mascara_parcelas -> rasterización del shapefile (identidad de parcela)
# Un píxel puede caer dentro del polígono de una parcela de algodón y
# llevar, sin embargo, el código de trigo en el cropmap.

datos_por_cultivo = {}
pixeles_por_parcela = defaultdict(lambda: defaultdict(int))

for cultivo, etiqueta_num in clases_cultivo.items():

    if cultivo == "None":
        continue

    ruta_completa = os.path.join(
        ruta_carpeta_npz,
        f"{cultivo}_series_2017.npz"
    )

    if not os.path.exists(ruta_completa):
        print(f"⚠ No existe {ruta_completa}")
        continue

    datos_npz = np.load(ruta_completa)

    vv = datos_npz["VV"]
    vh = datos_npz["VH"]
    
    #coh_vv = datos_npz["COH_VV"]
    #coh_vh = datos_npz["COH_VH"]

    #ndvi = datos_npz["NDVI"]  # Si se desea incluir NDVI, descomentar esta línea y agregarla a la pila de características
    #nir = datos_npz["NIR"]  # Si se desea incluir NIR, descomentar esta línea y agregarla a la pila de características
    
    coords = datos_npz["coords"]
    datos_npz.close()

    todo_el_cultivo_pixeles = np.concatenate(
        [
            vv,
            vh
            #coh_vv,
            #coh_vh,
            # ndvi, # Si se desea incluir NDVI, descomentar esta línea y agregarla a la pila de características
            #nir  # Si se desea incluir NIR, descomentar esta línea y agregarla a la pila de características
        ],
        axis=-1
    )

    filas_img = coords[:, 0]
    cols_img = coords[:, 1]

    ids_parcelas_por_pixel = mascara_parcelas[filas_img, cols_img]

    mask_valid = ids_parcelas_por_pixel != 0

    ids_validos = ids_parcelas_por_pixel[mask_valid]
    pixeles_validos = todo_el_cultivo_pixeles[mask_valid]
    coords_validos = coords[mask_valid]

    if len(np.unique(ids_validos)) < 2:
        print(f"⚠ {cultivo}: menos de dos parcelas.")
        continue

    # Guardamos ya filtrado para la Pasada 2, sin releer el .npz
    datos_por_cultivo[cultivo] = {
        "X": pixeles_validos,
        "coords": coords_validos,
        "parcelas": ids_validos,
        "etiqueta_num": etiqueta_num
    }

    ids_unicos, conteos = np.unique(ids_validos, return_counts=True)
    for pid, n in zip(ids_unicos, conteos):
        pixeles_por_parcela[pid][cultivo] += int(n)

n_multi = sum(1 for d in pixeles_por_parcela.values() if len(d) > 1)
print(f"\nParcelas únicas totales: {len(pixeles_por_parcela)}")
print(f"Parcelas con píxeles en más de un cultivo (desajuste cropmap/shapefile): {n_multi}")


# ============================================================
# CULTIVO DOMINANTE POR PARCELA
# ============================================================
# El cultivo dominante se calcula sobre TODOS los píxeles de la parcela,
# antes de filtrar: es la mejor estimación disponible de qué cultivo
# ocupa realmente el polígono.

cultivo_dominante = {pid: max(d, key=d.get) for pid, d in pixeles_por_parcela.items()}

print("\n" + "=" * 78)
print("REASIGNACIÓN DE PÍXELES AL CULTIVO DOMINANTE DE SU PARCELA")
print("=" * 78)
 
acum = {c: {"X": [], "coords": [], "parcelas": []} for c in datos_por_cultivo}
descartes_por_cultivo = {}          # ahora: píxeles reasignados HACIA cada cultivo
n_descartados_total = 0             # ahora: total de píxeles reasignados
n_totales_previo = 0
 
for cultivo, datos in datos_por_cultivo.items():
    ids = datos["parcelas"]
    destino = np.array([cultivo_dominante[pid] for pid in ids])
    n_totales_previo += len(ids)
 
    for c_dest in np.unique(destino):
        m = destino == c_dest
        if c_dest not in acum:                       # cultivo dominante sin datos propios
            acum[c_dest] = {"X": [], "coords": [], "parcelas": []}
        acum[c_dest]["X"].append(datos["X"][m])
        acum[c_dest]["coords"].append(datos["coords"][m])
        acum[c_dest]["parcelas"].append(ids[m])
        if c_dest != cultivo:
            n = int(m.sum())
            descartes_por_cultivo[c_dest] = descartes_por_cultivo.get(c_dest, 0) + n
            n_descartados_total += n
 
datos_por_cultivo = {
    c: {
        "X": np.concatenate(a["X"]),
        "coords": np.concatenate(a["coords"]),
        "parcelas": np.concatenate(a["parcelas"]),
        "etiqueta_num": clases_cultivo[c],
    }
    for c, a in acum.items() if len(a["X"]) > 0
}
 
print(f"Píxeles reasignados: {n_descartados_total} de {n_totales_previo} "
      f"({100 * n_descartados_total / n_totales_previo:.2f}%)")
 

# Descartar cultivos que se hayan quedado sin datos utilizables
for cultivo in [c for c, d in datos_por_cultivo.items()
                if len(np.unique(d["parcelas"])) < 2]:
    print(f"⚠ {cultivo}: menos de dos parcelas tras el filtro, se excluye.")
    del datos_por_cultivo[cultivo]

# ============================================================
# TAMAÑO POR PARCELA (recalculado tras el filtro)
# ============================================================
# El balanceo del split debe hacerse sobre los píxeles que realmente se
# van a usar, no sobre los originales.

tamano_total_parcela = defaultdict(int)
for cultivo, datos in datos_por_cultivo.items():
    ids_unicos, conteos = np.unique(datos["parcelas"], return_counts=True)
    for pid, n in zip(ids_unicos, conteos):
        tamano_total_parcela[pid] += int(n)

parcelas_por_cultivo_dominante = defaultdict(list)
for pid in tamano_total_parcela:
    parcelas_por_cultivo_dominante[cultivo_dominante[pid]].append(pid)

print(f"\nParcelas utilizables tras el filtro: {len(tamano_total_parcela)}")

# ============================================================
# SPLIT GLOBAL POR PARCELA (50/50), UNA SOLA DECISIÓN POR PARCELA
# ============================================================

parcela_split = {}

for cultivo, parcelas_del_cultivo in parcelas_por_cultivo_dominante.items():

    if len(parcelas_del_cultivo) < 2:
        print(f"⚠ {cultivo}: menos de dos parcelas dominantes, todas a train.")
        for pid in parcelas_del_cultivo:
            parcela_split[pid] = "train"
        continue

    # Mezcla reproducible (para desempates)
    parcelas_arr = RNG.permutation(parcelas_del_cultivo)

    # Ordenar de mayor a menor superficie TOTAL utilizable
    parcelas_ordenadas = sorted(
        parcelas_arr,
        key=lambda p: tamano_total_parcela[p],
        reverse=True
    )

    pixeles_train_acum = 0
    pixeles_test_acum = 0

    # Greedy ponderado hacia 50/50
    for parcela in parcelas_ordenadas:

        area = tamano_total_parcela[parcela]

        ratio_train = pixeles_train_acum / PROP_TRAIN
        ratio_test = pixeles_test_acum / PROP_TEST

        if ratio_train <= ratio_test:
            parcela_split[parcela] = "train"
            pixeles_train_acum += area
        else:
            parcela_split[parcela] = "test"
            pixeles_test_acum += area

# ============================================================
# PASADA 2 — APLICAR EL SPLIT YA DECIDIDO A CADA CULTIVO
# ============================================================

for cultivo, datos in list(datos_por_cultivo.items()):

    etiqueta_num = datos["etiqueta_num"]
    ids_validos = datos["parcelas"]
    pixeles_validos = datos["X"]
    coords_validos = datos["coords"]

    parcelas_unicas = np.unique(ids_validos)

    split_por_pixel = np.array([parcela_split[pid] for pid in ids_validos])

    mask_train = split_por_pixel == "train"
    mask_test = split_por_pixel == "test"

    pixeles_train = pixeles_validos[mask_train]
    pixeles_test = pixeles_validos[mask_test]

    coords_train = coords_validos[mask_train]
    coords_test = coords_validos[mask_test]

    ids_train = ids_validos[mask_train]
    ids_test = ids_validos[mask_test]

    # ========================================================
    # Guardado estructurado
    # ========================================================

    datos_train_por_cultivo[cultivo] = {
        "X": pixeles_train,
        "coords": coords_train,
        "parcelas": ids_train
    }

    datos_test_por_cultivo[cultivo] = {
        "X": pixeles_test,
        "coords": coords_test,
        "parcelas": ids_test
    }

    # ========================================================
    # Reporte
    # ========================================================

    reporte_division[cultivo] = {

        "parc_totales": len(parcelas_unicas),

        "parc_train": len(np.unique(ids_train)),

        "parc_test": len(np.unique(ids_test)),

        "pix_totales": len(pixeles_validos),

        "pix_train": len(pixeles_train),

        "pix_test": len(pixeles_test),

        "pix_descartados": descartes_por_cultivo.get(cultivo, 0),

        "balance_pixeles": abs(
            len(pixeles_train) - len(pixeles_test)
        )

    }


# ============================================================
# VERIFICACIONES
# ============================================================

# 1) Ninguna parcela puede quedar repartida entre train y test
verificacion = defaultdict(set)
for cultivo, datos in datos_train_por_cultivo.items():
    for pid in np.unique(datos['parcelas']):
        verificacion[pid].add("train")
for cultivo, datos in datos_test_por_cultivo.items():
    for pid in np.unique(datos['parcelas']):
        verificacion[pid].add("test")

solapadas = sum(1 for s in verificacion.values() if len(s) > 1)
print(f"\nParcelas solapadas entre train y test (debe ser 0): {solapadas}")

# 2) Ninguna parcela puede aparecer ya en más de un cultivo
apariciones = defaultdict(set)
for cultivo, datos in datos_por_cultivo.items():
    for pid in np.unique(datos["parcelas"]):
        apariciones[pid].add(cultivo)

multi_cultivo = sum(1 for s in apariciones.values() if len(s) > 1)
print(f"Parcelas presentes en más de un cultivo (debe ser 0): {multi_cultivo}")

# ============================================================
# COMPROBACIÓN
# ============================================================

print("\nFormato de los datos almacenados:\n")

for clase, datos in datos_train_por_cultivo.items():

    print(
        f"{clase:<15} "
        f"X:{datos['X'].shape} "
        f"Coords:{datos['coords'].shape} "
        f"Parcelas:{len(np.unique(datos['parcelas']))}"
    )

# ============================================================
# REPORTE FINAL
# ============================================================

print("\n")
print("=" * 120)
print(
    f"{'Cultivo':<15}"
    f"{'Parcelas(T/T)':>18}"
    f"{'Pixeles(T/T)':>25}"
    f"{'%Train':>12}"
    f"{'Dif.Pix':>12}"
    f"{'Descart.':>12}"
)
print("=" * 120)

for cultivo, info in reporte_division.items():

    porcentaje = (
        info["pix_train"] /
        info["pix_totales"] * 100
    )

    print(
        f"{cultivo:<15}"
        f"{info['parc_train']:>6}/{info['parc_test']:<6}"
        f"{info['pix_train']:>12}/{info['pix_test']:<12}"
        f"{porcentaje:>10.2f}%"
        f"{info['balance_pixeles']:>12}"
        f"{info['pix_descartados']:>12}"
    )

print("=" * 120)
print(f"{'TOTAL DESCARTADO':<15}{n_descartados_total:>96}")
print("=" * 120)

# ============================================================
# GUARDADO
# ============================================================

torch.save(
    {
        "train": datos_train_por_cultivo,
        "test": datos_test_por_cultivo
    },
    "/home/angelo/MyStore/Proyecto Investigador/2017/dataset_split_2017/dataset_split_2017_(backscatter)_v1.pt"
)

print("\n Dataset guardado correctamente (split sin solape train/test).")

⚠ No existe /home/angelo/MyStore/Proyecto Investigador/2017/series_temporales_2017_normalized(backscatter)/Barley_series_2017.npz
⚠ No existe /home/angelo/MyStore/Proyecto Investigador/2017/series_temporales_2017_normalized(backscatter)/Oats_series_2017.npz
⚠ Rice: menos de dos parcelas.

Parcelas únicas totales: 3547
Parcelas con píxeles en más de un cultivo (desajuste cropmap/shapefile): 0

REASIGNACIÓN DE PÍXELES AL CULTIVO DOMINANTE DE SU PARCELA
Píxeles reasignados: 0 de 1137929 (0.00%)

Parcelas utilizables tras el filtro: 3547

Parcelas solapadas entre train y test (debe ser 0): 0
Parcelas presentes en más de un cultivo (debe ser 0): 0

Formato de los datos almacenados:

Alfalfa         X:(15860, 122) Coords:(15860, 2) Parcelas:40
Carrot          X:(7925, 122) Coords:(7925, 2) Parcelas:36
Chickpea        X:(1743, 122) Coords:(1743, 2) Parcelas:12
Cotton          X:(238443, 122) Coords:(238443, 2) Parcelas:662
Fallow          X:(2756, 122) Coords:(2756, 2) Parcelas:36
Maize      

In [28]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
RUTA_DATASET_ENTRADA = "/home/angelo/MyStore/Proyecto Investigador/2017/dataset_split_2017/dataset_split_2017_(backscatter)_v1.pt"
RUTA_DATASET_SALIDA  = "/home/angelo/MyStore/Proyecto Investigador/2017/dataset_split_2017/dataset_split_2017_RF_(backscatter)_v1.pt"
SEED = 42

# ============================================================
# 1. CARGAR EL DATASET ORIGINAL (mismo split que usa el CAE)
# ============================================================
data = torch.load(RUTA_DATASET_ENTRADA, weights_only=False)
datos_train_por_cultivo = data["train"]
datos_test_por_cultivo  = data["test"]

# ============================================================
# 2. TAMAÑOS DE TRAIN POR CLASE Y CLASE MINORITARIA
# ============================================================
tam_train_por_clase = {
    clase: len(datos["X"])
    for clase, datos in datos_train_por_cultivo.items()
}

n_min             = min(tam_train_por_clase.values())
clase_minoritaria = min(tam_train_por_clase, key=tam_train_por_clase.get)

print("=" * 70)
print("TAMAÑOS DE TRAIN POR CLASE (ANTES DE BALANCEAR)")
print("=" * 70)
for clase, n in sorted(tam_train_por_clase.items(), key=lambda kv: kv[1]):
    print(f"{clase:15s} {n:8d} píxeles")
print("-" * 70)
print(f"Clase minoritaria: {clase_minoritaria} con {n_min} píxeles")
print(f"Todas las clases se recortarán (undersampling) a {n_min} píxeles en train")
print("=" * 70)

# ============================================================
# 3. BALANCEAR EL TRAIN (undersampling al tamaño de la clase minoritaria)
# ============================================================
rng = np.random.default_rng(SEED)

datos_train_balanceado = {}
resumen_balanceo       = {}

for clase, datos in datos_train_por_cultivo.items():

    X_clase       = datos["X"]
    coords_clase  = datos["coords"]
    parcelas_clase = datos["parcelas"]

    n_disponibles = len(X_clase)

    idx = np.arange(n_disponibles)
    rng.shuffle(idx)
    idx_seleccionados = idx[:n_min]

    datos_train_balanceado[clase] = {
        "X":        X_clase[idx_seleccionados],
        "coords":   coords_clase[idx_seleccionados],
        "parcelas": parcelas_clase[idx_seleccionados]
    }

    resumen_balanceo[clase] = {
        "disponibles": n_disponibles,
        "usados":      len(idx_seleccionados),
        "descartados": n_disponibles - len(idx_seleccionados)
    }

print("\n" + "=" * 70)
print("RESUMEN DEL BALANCEO (TRAIN)")
print("=" * 70)
for clase, info in resumen_balanceo.items():
    print(
        f"{clase:15s} "
        f"disponibles={info['disponibles']:8d} "
        f"usados={info['usados']:8d} "
        f"descartados={info['descartados']:8d}"
    )
print("-" * 70)
print(
    f"Total train balanceado: {n_min * len(resumen_balanceo)} muestras "
    f"({len(resumen_balanceo)} clases x {n_min} c/u)"
)
print("=" * 70)

# ============================================================
# 4. TEST: SE COPIA TAL CUAL, SIN TOCAR
# ============================================================
# El test no se balancea para que la evaluación refleje
# la proporción real de clases.

# ============================================================
# 5. GUARDAR EL NUEVO DATASET, INDEPENDIENTE DEL ORIGINAL
# ============================================================
os.makedirs(os.path.dirname(RUTA_DATASET_SALIDA), exist_ok=True)

torch.save(
    {
        "train": datos_train_balanceado,
        "test":  datos_test_por_cultivo,
        "meta":  {
            "n_min_clase":       int(n_min),
            "clase_minoritaria": clase_minoritaria,
            "seed_balanceo":     SEED,
            "dataset_origen":    RUTA_DATASET_ENTRADA,
            "resumen_balanceo":  resumen_balanceo
        }
    },
    RUTA_DATASET_SALIDA
)

print(f"\n✅ Dataset balanceado para RF guardado en {RUTA_DATASET_SALIDA}")
print("   (train balanceado, test intacto, con metadatos del balanceo incluidos)")

TAMAÑOS DE TRAIN POR CLASE (ANTES DE BALANCEAR)
Potato              1361 píxeles
Chickpea            1743 píxeles
Pepper              2129 píxeles
Fallow              2756 píxeles
Pumpkin             3589 píxeles
Quinoa              3959 píxeles
SweetPotato         4465 píxeles
Onion               5426 píxeles
Carrot              7925 píxeles
Alfalfa            15860 píxeles
Wheat              16783 píxeles
Sunflower          24342 píxeles
Maize              25367 píxeles
SugarBeet          83536 píxeles
Tomato            131278 píxeles
Cotton            238443 píxeles
----------------------------------------------------------------------
Clase minoritaria: Potato con 1361 píxeles
Todas las clases se recortarán (undersampling) a 1361 píxeles en train

RESUMEN DEL BALANCEO (TRAIN)
Alfalfa         disponibles=   15860 usados=    1361 descartados=   14499
Carrot          disponibles=    7925 usados=    1361 descartados=    6564
Chickpea        disponibles=    1743 usados=    1361 descarta